In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Task 5: Mental Health Support Chatbot (Fine-Tuned)

**Objective:** Build a basic chatbot that provides supportive and empathetic responses for stress, anxiety, and emotional wellness.
**Model Base:** DistilGPT2
**Dataset:** Empathetic Dialogues (Facebook AI)

**Goals for this Notebook:**
1. Load the Empathetic Dialogues dataset.
2. Fine-tune the `DistilGPT2` model using Hugging Face's `Trainer` API.
3. Ensure the tone is gentle and emotionally supportive.
4. Build a simple command-line interface (CLI) to test the chatbot.

In [ ]:
# 1. Install Required Libraries (No downgrading this time! Clean install)
!pip install transformers datasets accelerate -q

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch

# ==========================================
# 2. Dataset Loading & Preprocessing
# ==========================================
print("Loading Empathetic Dialogues dataset (Fixed Parquet Version)... 📚")

# 🎯 THE ULTIMATE FIX:
# Using the Parquet version of the exact same dataset to completely bypass the script error!
dataset = load_dataset("Ahren09/empathetic_dialogues")

# For fast training in Colab, taking a tiny subset (100 rows).
train_data = dataset['train'].select(range(100))

# Load Model & Tokenizer (DistilGPT2)
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

# Preprocessing function
def tokenize_function(examples):
    texts = [f"User: {u}\nBot:" for u in examples['utterance']]
    return tokenizer(texts, padding="max_length", truncation=True, max_length=64)

tokenized_train = train_data.map(tokenize_function, batched=True)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ==========================================
# 3. Model Fine-Tuning (Trainer API)
# ==========================================
print("Setting up Trainer API... ⚙️")
training_args = TrainingArguments(
    output_dir="./empathetic-bot",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_steps=50,
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator
)

print("Starting Fine-Tuning! This might take a minute... ⏳")
trainer.train()

# ==========================================
# 4. Command-Line Interface (CLI) for Testing (Improved Output)
# ==========================================
print("\n--- 🤖 Empathetic Chatbot CLI ---")
def chat_with_bot(prompt):
    inputs = tokenizer(f"User: {prompt}\nBot:", return_tensors="pt").to(model.device)

    # Generation parameters tweaked to stop repetition
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.7,
        do_sample=True,
        repetition_penalty=1.2,       # Stops repeating same words
        no_repeat_ngram_size=2        # Stops repeating phrases
    )

    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the Bot's first reply to keep it clean
    clean_reply = full_response.split("Bot:")[-1].strip()
    return clean_reply

# Test the chatbot
test_query = "I am feeling very stressed about my upcoming exams and I can't sleep."
print(f"👤 User: {test_query}")
print(f"🤖 Bot: {chat_with_bot(test_query)}")

Loading Empathetic Dialogues dataset (Fixed Parquet Version)... 📚


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setting up Trainer API... ⚙️
Starting Fine-Tuning! This might take a minute... ⏳


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,4.090498
20,3.428396


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- 🤖 Empathetic Chatbot CLI ---
👤 User: I am feeling very stressed about my upcoming exams and I can't sleep.
🤖 Bot: What should you do? (laugh) bot: How did they know that?? Bot :
